# Cálculo de métricas de calidad · ORCA

Mide las seis dimensiones de la plantilla (completitud, validez de rango, validez de lista, unicidad, consistencia y oportunidad) sobre **todo** el lote curado de DS-01: las 76 columnas que conserva el pipeline, no una muestra.

**Entrada:** `data/curated/DS-01_contratos_curated.parquet`  
**Salida:** `documentacion/calidad/metricas_por_columna_DS-01.csv` y `documentacion/calidad/metricas_reglas_DS-01.csv`

Este cuaderno no reemplaza el quality gate de `calidad.py`. Describe cada columna. El gate sigue siendo la decisión `PUBLICAR` / `CUARENTENA` de las 14 reglas aprobables por el Data Steward.

| Dimensión | Dónde aplica |
| --- | --- |
| Completitud | Las 76 columnas. Vacíos y registros con valor. Además cuenta dos textos que `a_nulos` no convierte: `No aplica/No pertenece` y `No Especificado`. |
| Unicidad | Las 76 columnas, como conteo de filas cuyo valor aparece una sola vez. La regla de calidad sigue siendo solo `id_contrato`. |
| Validez de rango | Columnas numéricas y de fecha, con el umbral de cada una. |
| Validez de lista | Catálogos cerrados de este corte, NIT, código DIVIPOLA y ciudad reconocida en el maestro. |
| Consistencia | Reglas entre dos campos. No es una medida de una sola columna. |
| Oportunidad | Columnas de fecha: cuántas caen desde el 1 de enero de 2026. La regla del lote es O1. |


## 1. Configuración

La raíz se detecta igual que en el EDA, para correr el cuaderno desde el proyecto o desde `notebooks/`.


In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

CURATED = ROOT / "data" / "curated"
SALIDA = ROOT / "documentacion" / "calidad"
SALIDA.mkdir(parents=True, exist_ok=True)

CORTE = pd.Timestamp("2026-10-23")
DESDE = pd.Timestamp("2026-01-01")
FIRMA_MIN = pd.Timestamp("2024-01-01")
VALOR_ATIPICO = 100_000_000_000
DIAS_PRORROGA_MAX = 3650
ANIO_MIN, ANIO_MAX = 1990, 2100
VENTANA_OPORTUNIDAD_DIAS = 7

VACIOS_DISFRAZADOS = {"no aplica/no pertenece", "no especificado"}

CATALOGOS = {
    "departamento": {"Antioquia"},
    "orden": {"Corporación Autónoma", "Nacional", "Territorial"},
    "rama": {"Corporación Autónoma", "Ejecutivo", "Judicial", "Legislativo"},
    "entidad_centralizada": {"Centralizada", "Descentralizada"},
    "estado_contrato": {
        "Aprobado", "Cedido", "Cerrado", "En ejecución", "Modificado", "Suspendido", "Terminado",
    },
    "origen_de_los_recursos": {"Distribuido", "Recursos Propios"},
    "destino_gasto": {"Funcionamiento", "Inversión"},
    "documentos_tipo": {"No", "Si"},
    "criterios_de_sostenibilidad_ambiental": {"No", "Sí"},
    "obligaciones_posconsumo": {"No", "Sí"},
    "sector": {
        "Ambiente y Desarrollo Sostenible", "Ciencia Tecnología", "Cultura", "Educación Nacional",
        "Hacienda y Crédito Público", "Industria", "Información Estadística", "Ley de Justicia",
        "No aplica/No pertenece", "Planeación", "Presidencia de la República",
        "Salud y Protección Social", "Servicio Público",
        "Tecnologías de la Información y las Comunicaciones", "Trabajo", "Transporte",
        "Vivienda, Ciudad y Territorio", "agricultura", "defensa", "deportes", "interior",
    },
    "tipo_de_contrato": {
        "Arrendamiento de inmuebles", "Arrendamiento de muebles", "Asociación Público Privada",
        "Comodato", "Compraventa", "Concesión", "Consultoría", "Decreto 092 de 2017",
        "Interventoría", "Negocio fiduciario", "No Especificado", "Obra",
        "Operaciones de Crédito Público", "Otro", "Prestación de servicios", "Seguros",
        "Servicios financieros", "Suministros", "Venta inmuebles", "Venta muebles",
    },
    "modalidad_de_contratacion": {
        "Concurso de méritos abierto", "Contratación Directa (con ofertas)", "Contratación directa",
        "Contratación régimen especial", "Contratación régimen especial (con ofertas)",
        "Enajenación de bienes con sobre cerrado", "Enajenación de bienes con subasta",
        "Licitación pública", "Licitación pública Obra Publica", "Mínima cuantía",
        "Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes",
        "Selección Abreviada de Menor Cuantía", "Selección abreviada subasta inversa",
    },
}

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


def miles(n):
    return f"{n:,}".replace(",", ".")


def pct(cumplen, evaluados):
    if not evaluados:
        return None
    return round(100 * cumplen / evaluados, 4)


print("Raíz:", ROOT)
print("Curated:", CURATED)


Raíz: C:\Users\Usuario\Documents\Universidad UPB\Sexto-septimo Semestre\Data Office\Proyecto final\DataOfficeStrategy_Project_202620UPB
Curated: C:\Users\Usuario\Documents\Universidad UPB\Sexto-septimo Semestre\Data Office\Proyecto final\DataOfficeStrategy_Project_202620UPB\data\curated


## 2. Carga del lote curado

Se lee el Parquet completo. No se vuelve a limpiar ni se llama a la API.


In [2]:
ruta = CURATED / "DS-01_contratos_curated.parquet"
df = pd.read_parquet(ruta)
n_filas, n_columnas = df.shape

print(f"{miles(n_filas)} filas, {n_columnas} columnas")
print("Archivo:", ruta.name)


291.719 filas, 76 columnas
Archivo: DS-01_contratos_curated.parquet


## 3. Completitud, unicidad y validez por columna

Una fila por columna.

* **Vacíos** es el equivalente de `CONTAR.BLANCO`: celdas nulas. En curated, `No Definido`, `N/A` y `Sin descripción` ya son nulos.
* **Con valor** es el equivalente de `CONTARA`.
* **Vacíos disfrazados** son los textos `No aplica/No pertenece` y `No Especificado`, que siguen guardados como categoría.
* **Filas con valor único** es el equivalente de la fórmula de unicidad: filas cuyo valor aparece una sola vez. En `ciudad` o `modalidad_de_contratacion` la repetición es el comportamiento esperado. La regla de calidad es `id_contrato`.
* **Validez de rango.** Montos: no negativos. `valor_del_contrato`: además, no mayor a 100.000 millones. `dias_adicionados`: entre 0 y 3.650, el umbral que ya usa el cuaderno de limpieza. `fecha_de_firma`: entre el 1 de enero de 2024 y el corte. Las demás fechas: año entre 1990 y 2100, para marcar años como 0025 o 2925.
* **Validez de lista.** Los catálogos de arriba son el dominio observado en el corte del 23 de octubre de 2026, escrito en el cuaderno para que un lote posterior pueda fallar. `nit_entidad` debe tener 9 dígitos. `proveedor_nit`, cuando existe, 9 o 10 dígitos. `cod_municipio`, cuando existe, 5 dígitos. `ciudad`, cuando existe, debe tener código DIVIPOLA.


In [3]:
def vacios_disfrazados(serie):
    if str(serie.dtype) != "string" and serie.dtype != object:
        return 0
    texto = serie.dropna().astype(str).str.strip().str.lower()
    return int(texto.isin(VACIOS_DISFRAZADOS).sum())


def unicidad(serie):
    presentes = serie.dropna()
    if presentes.empty:
        return 0, 0, 0
    conteo = presentes.value_counts()
    filas_unicas = int(conteo[conteo == 1].sum())
    filas_repetidas = int(conteo[conteo > 1].sum())
    return int(conteo.size), filas_unicas, filas_repetidas


def validez_rango(columna, serie):
    tipo = str(serie.dtype)
    presentes = serie.notna()
    evaluados = int(presentes.sum())
    if tipo.startswith(("Float", "float", "int", "Int")):
        if columna == "dias_adicionados":
            falla = presentes & ((serie < 0) | (serie > DIAS_PRORROGA_MAX))
            return "0 a 3.650 días", evaluados, int(falla.sum())
        if columna == "valor_del_contrato":
            falla = presentes & ((serie < 0) | (serie > VALOR_ATIPICO))
            return "0 a 100.000 millones", evaluados, int(falla.sum())
        falla = presentes & (serie < 0)
        return "mayor o igual a 0", evaluados, int(falla.sum())
    if tipo.startswith("datetime"):
        if columna == "fecha_de_firma":
            falla = presentes & ((serie < FIRMA_MIN) | (serie > CORTE))
            return "entre 2024-01-01 y el corte", evaluados, int(falla.sum())
        anio = serie.dt.year
        falla = presentes & ((anio < ANIO_MIN) | (anio > ANIO_MAX))
        return f"año entre {ANIO_MIN} y {ANIO_MAX}", evaluados, int(falla.sum())
    return None, None, None


def validez_lista(columna, serie):
    presentes = serie.notna()
    evaluados = int(presentes.sum())
    if columna in CATALOGOS:
        falla = presentes & ~serie.isin(CATALOGOS[columna])
        return "catálogo del corte 2026-10-23", evaluados, int(falla.sum())
    if columna == "nit_entidad":
        falla = presentes & ~serie.astype("string").str.fullmatch(r"\d{9}").fillna(False)
        return "9 dígitos", evaluados, int(falla.sum())
    if columna == "proveedor_nit":
        falla = presentes & ~serie.astype("string").str.fullmatch(r"\d{9}|\d{10}").fillna(False)
        return "9 o 10 dígitos", evaluados, int(falla.sum())
    if columna == "cod_municipio":
        falla = presentes & ~serie.astype("string").str.fullmatch(r"\d{5}").fillna(False)
        return "código DIVIPOLA de 5 dígitos", evaluados, int(falla.sum())
    if columna == "ciudad":
        falla = presentes & df["cod_municipio"].isna()
        return "ciudad reconocida en DIVIPOLA Antioquia", evaluados, int(falla.sum())
    if tipo_booleano(serie):
        return "verdadero o falso", evaluados, 0
    return None, None, None


def tipo_booleano(serie):
    return str(serie.dtype).startswith("bool")


filas = []
for columna in df.columns:
    serie = df[columna]
    vacios = int(serie.isna().sum())
    disfrazados = vacios_disfrazados(serie)
    con_valor = n_filas - vacios
    distintos, filas_unicas, filas_repetidas = unicidad(serie)
    rango, rango_eval, rango_fallan = validez_rango(columna, serie)
    lista, lista_eval, lista_fallan = validez_lista(columna, serie)
    es_fecha = str(serie.dtype).startswith("datetime")
    filas.append({
        "columna": columna,
        "tipo": str(serie.dtype),
        "filas": n_filas,
        "vacios": vacios,
        "con_valor": con_valor,
        "completitud_pct": pct(con_valor, n_filas),
        "vacios_disfrazados": disfrazados,
        "completitud_estricta_pct": pct(con_valor - disfrazados, n_filas),
        "valores_distintos": distintos,
        "filas_con_valor_unico": filas_unicas,
        "filas_con_valor_repetido": filas_repetidas,
        "validez_rango": rango,
        "rango_evaluados": rango_eval,
        "rango_fallan": rango_fallan,
        "rango_cumplimiento_pct": pct(rango_eval - rango_fallan, rango_eval) if rango else None,
        "validez_lista": lista,
        "lista_evaluados": lista_eval,
        "lista_fallan": lista_fallan,
        "lista_cumplimiento_pct": pct(lista_eval - lista_fallan, lista_eval) if lista else None,
        "fechas_desde_2026_01_01": int((serie >= DESDE).sum()) if es_fecha else None,
        "fechas_posteriores_al_corte": int((serie > CORTE).sum()) if es_fecha else None,
    })

por_columna = pd.DataFrame(filas)
print(f"Métricas calculadas para {len(por_columna)} columnas y {miles(n_filas)} filas.")


Métricas calculadas para 76 columnas y 291.719 filas.


### 3.1 Completitud

La tabla queda ordenada por la completitud estricta, de menor a mayor, para que los vacíos disfrazados no queden al final. El archivo CSV conserva el orden de las columnas del Parquet.

`proveedor_nit` y `proveedor_nombre` quedan vacíos a propósito cuando el proveedor es persona natural: ese vacío es el diseño del pipeline, no un campo que la fuente dejó en blanco. Lo mismo ocurre con los seudónimos cuando el documento de origen no vino.


In [4]:
completitud = por_columna[
    ["columna", "tipo", "vacios", "con_valor", "completitud_pct", "vacios_disfrazados", "completitud_estricta_pct"]
].sort_values(["completitud_estricta_pct", "vacios"], ascending=[True, False])
completitud


,columna,tipo,vacios,con_valor,completitud_pct,vacios_disfrazados,completitud_estricta_pct
56,criterios_de_sostenibilidad_ambiental,string,286995,4724,1.6194,0,1.6194
60,fecha_de_notificaci_n_de_prorrogaci_n,datetime64[us],273834,17885,6.1309,0,6.1309
59,fecha_fin_liquidacion,datetime64[us],246525,45194,15.4923,0,15.4923
58,fecha_inicio_liquidacion,datetime64[us],246521,45198,15.4937,0,15.4937
69,proveedor_nit,string,237091,54628,18.7262,0,18.7262
70,proveedor_nombre,string,236995,54724,18.7591,0,18.7591
72,representante_legal_id,string,211602,80117,27.4638,0,27.4638
75,ordenador_pago_id,string,133322,158397,54.2978,0,54.2978
50,ultima_actualizacion,datetime64[us],130232,161487,55.3570,0,55.3570
21,condiciones_de_entrega,string,96881,194838,66.7896,0,66.7896


### 3.2 Unicidad

`filas_con_valor_unico` cuenta las filas cuyo valor no se repite. En un identificador, ese número debería acercarse al total de filas. En una categoría, casi todas las filas se repiten y el conteo bajo no es un defecto.


In [5]:
unicidad_tabla = por_columna[
    ["columna", "con_valor", "valores_distintos", "filas_con_valor_unico", "filas_con_valor_repetido"]
]
unicidad_tabla


,columna,con_valor,valores_distintos,filas_con_valor_unico,filas_con_valor_repetido
0,nombre_entidad,291719,414,62,291657
1,nit_entidad,291719,414,62,291657
2,departamento,291719,1,0,291719
3,ciudad,272024,118,4,272020
4,localizaci_n,291719,133,4,291715
5,orden,291719,3,0,291719
6,sector,291719,21,0,291719
7,rama,291719,4,0,291719
8,entidad_centralizada,291719,2,0,291719
9,proceso_de_compra,291719,281959,280175,11544


### 3.3 Validez de rango y de lista

Solo aparecen las columnas que tienen una regla. Una celda nula no entra al denominador: la ausencia ya quedó en completitud.


In [6]:
rango = por_columna.loc[
    por_columna["validez_rango"].notna(),
    ["columna", "validez_rango", "rango_evaluados", "rango_fallan", "rango_cumplimiento_pct"],
].sort_values("rango_fallan", ascending=False)

lista = por_columna.loc[
    por_columna["validez_lista"].notna(),
    ["columna", "validez_lista", "lista_evaluados", "lista_fallan", "lista_cumplimiento_pct"],
].sort_values("lista_fallan", ascending=False)

print("Validez de rango")
display(rango)
print("Validez de lista")
display(lista)


Validez de rango


,columna,validez_rango,rango_evaluados,rango_fallan,rango_cumplimiento_pct
34,valor_pendiente_de_pago,mayor o igual a 0,"291,719.0000",169.0000,99.9421
38,valor_pendiente_de_ejecucion,mayor o igual a 0,"291,719.0000",169.0000,99.9421
31,valor_del_contrato,0 a 100.000 millones,"291,719.0000",47.0000,99.9839
58,fecha_inicio_liquidacion,año entre 1990 y 2100,"45,198.0000",10.0000,99.9779
42,dias_adicionados,0 a 3.650 días,"291,719.0000",7.0000,99.9976
60,fecha_de_notificaci_n_de_prorrogaci_n,año entre 1990 y 2100,"17,885.0000",5.0000,99.9720
20,fecha_de_fin_del_contrato,año entre 1990 y 2100,"291,718.0000",1.0000,99.9997
45,sistema_general_de_participaciones,mayor o igual a 0,"291,719.0000",0.0000,100.0000
59,fecha_fin_liquidacion,año entre 1990 y 2100,"45,194.0000",0.0000,100.0000
50,ultima_actualizacion,año entre 1990 y 2100,"161,487.0000",0.0000,100.0000


Validez de lista


,columna,validez_lista,lista_evaluados,lista_fallan,lista_cumplimiento_pct
69,proveedor_nit,9 o 10 dígitos,"54,628.0000",657.0000,98.7973
3,ciudad,ciudad reconocida en DIVIPOLA Antioquia,"272,024.0000",629.0000,99.7688
29,origen_de_los_recursos,catálogo del corte 2026-10-23,"291,719.0000",0.0000,100.0000
68,proveedor_es_persona_natural,verdadero o falso,"291,719.0000",0.0000,100.0000
67,flag_sin_fecha_firma,verdadero o falso,"291,719.0000",0.0000,100.0000
66,flag_fechas_inconsistentes,verdadero o falso,"291,719.0000",0.0000,100.0000
65,flag_valor_atipico,verdadero o falso,"291,719.0000",0.0000,100.0000
64,flag_valor_cero,verdadero o falso,"291,719.0000",0.0000,100.0000
63,es_contratacion_directa,verdadero o falso,"291,657.0000",0.0000,100.0000
61,cod_municipio,código DIVIPOLA de 5 dígitos,"271,395.0000",0.0000,100.0000


## 4. Consistencia

Cada regla compara dos campos. Si a una fila le falta alguno de los dos, no entra al denominador.

K1 y K2 son las reglas del quality gate. K3 es el conteo que el cuaderno de limpieza deja en memoria y no escribe como columna: inicio anterior a la firma. K4 revisa la separación de persona natural. Las reglas B confirman que las banderas guardadas coinciden con la condición que las define.


In [7]:
firma = df["fecha_de_firma"]
inicio = df["fecha_de_inicio_del_contrato"]
fin = df["fecha_de_fin_del_contrato"]
persona = df["proveedor_es_persona_natural"].eq(True)
sin_organizacion = df["proveedor_nit"].isna() & df["proveedor_nombre"].isna()
modalidad = df["modalidad_de_contratacion"]
directa_esperada = modalidad.str.lower().str.startswith("contratación directa")


def regla(dimension, identificador, descripcion, ok):
    evaluados = int(ok.notna().sum())
    cumplen = int(ok.fillna(False).sum())
    return {
        "dimension": dimension,
        "id": identificador,
        "descripcion": descripcion,
        "evaluados": evaluados,
        "cumplen": cumplen,
        "fallan": evaluados - cumplen,
        "cumplimiento_pct": pct(cumplen, evaluados),
    }


reglas = [
    regla("Consistencia", "K1", "fecha de fin no anterior a la de inicio",
          (fin >= inicio).where(fin.notna() & inicio.notna())),
    regla("Consistencia", "K2", "fecha de firma no posterior al fin",
          (firma <= fin).where(firma.notna() & fin.notna())),
    regla("Consistencia", "K3", "fecha de inicio no anterior a la firma",
          (inicio >= firma).where(inicio.notna() & firma.notna())),
    regla("Consistencia", "K4", "persona natural sin NIT ni nombre de organización",
          sin_organizacion.where(persona)),
    regla("Consistencia", "B1", "es_contratacion_directa coincide con la modalidad",
          (df["es_contratacion_directa"] == directa_esperada).where(modalidad.notna())),
    regla("Consistencia", "B2", "flag_valor_cero coincide con valor igual a 0",
          df["flag_valor_cero"] == (df["valor_del_contrato"] == 0)),
    regla("Consistencia", "B3", "flag_valor_atipico coincide con valor mayor a 100.000 millones",
          df["flag_valor_atipico"] == (df["valor_del_contrato"] > VALOR_ATIPICO)),
    regla("Consistencia", "B4", "flag_fechas_inconsistentes coincide con fin anterior al inicio",
          df["flag_fechas_inconsistentes"].fillna(False) == (fin < inicio).fillna(False)),
    regla("Consistencia", "B5", "flag_sin_fecha_firma coincide con firma nula",
          df["flag_sin_fecha_firma"].fillna(False) == firma.isna()),
    regla("Unicidad", "U1", "id_contrato sin filas repetidas",
          pd.Series(~df["id_contrato"].duplicated(keep=False), index=df.index)),
]

consistencia = pd.DataFrame(reglas)
consistencia


,dimension,id,descripcion,evaluados,cumplen,fallan,cumplimiento_pct
0,Consistencia,K1,fecha de fin no anterior a la de inicio,289126,289041,85,99.9706
1,Consistencia,K2,fecha de firma no posterior al fin,291718,291456,262,99.9102
2,Consistencia,K3,fecha de inicio no anterior a la firma,289126,289115,11,99.9962
3,Consistencia,K4,persona natural sin NIT ni nombre de organización,236967,236967,0,100.0000
4,Consistencia,B1,es_contratacion_directa coincide con la modalidad,291657,291657,0,100.0000
5,Consistencia,B2,flag_valor_cero coincide con valor igual a 0,291719,291719,0,100.0000
6,Consistencia,B3,flag_valor_atipico coincide con valor mayor a 100.000 millones,291719,291719,0,100.0000
7,Consistencia,B4,flag_fechas_inconsistentes coincide con fin anterior al inicio,291719,291719,0,100.0000
8,Consistencia,B5,flag_sin_fecha_firma coincide con firma nula,291719,291719,0,100.0000
9,Unicidad,U1,id_contrato sin filas repetidas,291719,291719,0,100.0000


## 5. Oportunidad

Para cada fecha se cuenta cuántos valores caen desde el 1 de enero de 2026 y cuántos son posteriores al corte del 23 de octubre de 2026. Una fecha de fin posterior al corte puede ser un contrato vigente: ese conteo no es una falla.

O1 sí es una regla del lote: la firma más reciente tiene que caer dentro de los 7 días anteriores al corte.


In [8]:
oportunidad = por_columna.loc[
    por_columna["fechas_desde_2026_01_01"].notna(),
    ["columna", "con_valor", "fechas_desde_2026_01_01", "fechas_posteriores_al_corte"],
].copy()

ultima_firma = firma.max()
dentro_de_ventana = bool(
    pd.notna(ultima_firma)
    and ultima_firma >= CORTE - pd.Timedelta(days=VENTANA_OPORTUNIDAD_DIAS)
    and ultima_firma <= CORTE
)
reglas.append(regla(
    "Oportunidad",
    "O1",
    f"última firma ({ultima_firma.date()}) dentro de los {VENTANA_OPORTUNIDAD_DIAS} días anteriores al corte",
    pd.Series([dentro_de_ventana], index=[0]),
))
reglas_tabla = pd.DataFrame(reglas)

print(f"Última firma: {ultima_firma.date()} · corte: {CORTE.date()} · O1: {'cumple' if dentro_de_ventana else 'falla'}")
oportunidad


Última firma: 2026-10-20 · corte: 2026-10-23 · O1: cumple


,columna,con_valor,fechas_desde_2026_01_01,fechas_posteriores_al_corte
18,fecha_de_firma,291719,"99,108.0000",0.0000
19,fecha_de_inicio_del_contrato,289126,"98,551.0000",18.0000
20,fecha_de_fin_del_contrato,291718,"107,945.0000","56,690.0000"
50,ultima_actualizacion,161487,"78,152.0000",0.0000
58,fecha_inicio_liquidacion,45198,"20,110.0000","10,390.0000"
59,fecha_fin_liquidacion,45194,"30,223.0000","18,399.0000"
60,fecha_de_notificaci_n_de_prorrogaci_n,17885,"5,900.0000","2,826.0000"


## 6. Archivos

Se guardan dos CSV. El primero tiene una fila por columna. El segundo tiene las reglas entre campos y la regla O1.


In [9]:
ruta_columnas = SALIDA / "metricas_por_columna_DS-01.csv"
ruta_reglas = SALIDA / "metricas_reglas_DS-01.csv"
por_columna.to_csv(ruta_columnas, index=False, encoding="utf-8-sig")
reglas_tabla.to_csv(ruta_reglas, index=False, encoding="utf-8-sig")

con_vacios = int((por_columna["vacios"] > 0).sum())
disfrazados_total = int(por_columna["vacios_disfrazados"].sum())
rango_con_fallas = por_columna.loc[por_columna["rango_fallan"].fillna(0) > 0, ["columna", "validez_rango", "rango_fallan"]]
lista_con_fallas = por_columna.loc[por_columna["lista_fallan"].fillna(0) > 0, ["columna", "validez_lista", "lista_fallan"]]

print(f"Columnas con al menos un vacío: {con_vacios} de {n_columnas}")
print(f"Celdas con vacío disfrazado residual: {miles(disfrazados_total)}")
print("Rango con fallas:")
display(rango_con_fallas)
print("Lista con fallas:")
display(lista_con_fallas)
print("Reglas:")
display(reglas_tabla)
print("Escrito:", ruta_columnas)
print("Escrito:", ruta_reglas)


Columnas con al menos un vacío: 29 de 76
Celdas con vacío disfrazado residual: 54.020
Rango con fallas:


,columna,validez_rango,rango_fallan
20,fecha_de_fin_del_contrato,año entre 1990 y 2100,1.0000
31,valor_del_contrato,0 a 100.000 millones,47.0000
34,valor_pendiente_de_pago,mayor o igual a 0,169.0000
38,valor_pendiente_de_ejecucion,mayor o igual a 0,169.0000
42,dias_adicionados,0 a 3.650 días,7.0000
58,fecha_inicio_liquidacion,año entre 1990 y 2100,10.0000
60,fecha_de_notificaci_n_de_prorrogaci_n,año entre 1990 y 2100,5.0000


Lista con fallas:


,columna,validez_lista,lista_fallan
3,ciudad,ciudad reconocida en DIVIPOLA Antioquia,629.0000
69,proveedor_nit,9 o 10 dígitos,657.0000


Reglas:


,dimension,id,descripcion,evaluados,cumplen,fallan,cumplimiento_pct
0,Consistencia,K1,fecha de fin no anterior a la de inicio,289126,289041,85,99.9706
1,Consistencia,K2,fecha de firma no posterior al fin,291718,291456,262,99.9102
2,Consistencia,K3,fecha de inicio no anterior a la firma,289126,289115,11,99.9962
3,Consistencia,K4,persona natural sin NIT ni nombre de organización,236967,236967,0,100.0000
4,Consistencia,B1,es_contratacion_directa coincide con la modalidad,291657,291657,0,100.0000
5,Consistencia,B2,flag_valor_cero coincide con valor igual a 0,291719,291719,0,100.0000
6,Consistencia,B3,flag_valor_atipico coincide con valor mayor a 100.000 millones,291719,291719,0,100.0000
7,Consistencia,B4,flag_fechas_inconsistentes coincide con fin anterior al inicio,291719,291719,0,100.0000
8,Consistencia,B5,flag_sin_fecha_firma coincide con firma nula,291719,291719,0,100.0000
9,Unicidad,U1,id_contrato sin filas repetidas,291719,291719,0,100.0000


Escrito: C:\Users\Usuario\Documents\Universidad UPB\Sexto-septimo Semestre\Data Office\Proyecto final\DataOfficeStrategy_Project_202620UPB\documentacion\calidad\metricas_por_columna_DS-01.csv
Escrito: C:\Users\Usuario\Documents\Universidad UPB\Sexto-septimo Semestre\Data Office\Proyecto final\DataOfficeStrategy_Project_202620UPB\documentacion\calidad\metricas_reglas_DS-01.csv
